# 158 — Resiliencia, idempotencia, rollback y recuperación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Cotas: `min(10, 0.25·2^n)` para n = 0..3 →
**0.25, 0.5, 1, 2 s** (4 esperas para 5 intentos; tras el 5.º no se espera:
se falla en definitivo). Peor caso = 0.25+0.5+1+2 = **3.75 s**; media
acumulada = 3.75/2 = **1.875 s**. El tope de 10 s nunca se alcanza.

**Ejercicio 2.** F → cerrado (1 fallo); F → cerrado (2); É → cerrado (la
ventana sigue contando fallos recientes); F → **abierto** (3 fallos en
ventana); F, F → abierto (se rechazan sin tocar la dependencia); pasan 30 s →
**semiabierto**; prueba É → **cerrado**. Secuencia: cerrado, cerrado,
cerrado, abierto, abierto, abierto, semiabierto→cerrado.

**Ejercicio 3.**
a) **Sí**: asignación absoluta, N ejecuciones = mismo estado.
b) **No**: cada ejecución suma; reintentar duplica el efecto.
c) **Sí**: la clave K permite al servidor deduplicar y devolver el resultado
   original.
d) **No**: cada reintento puede enviar otro correo; necesita clave o registro
   de envío.
e) **Sí en efecto** (el recurso queda ausente), aunque la *respuesta* pueda
   cambiar (200 y luego 404): idempotencia de estado, no de código de
   respuesta.

**Ejercicio 4.** Falla T₄ → se ejecutan **C₃, C₂, C₁** (orden inverso; C₄ no
se ejecuta porque T₄ no llegó a completarse). La saga garantiza que, si las
compensaciones terminan, el estado converge al equivalente de «no hubo
reunión» (consistencia eventual). NO garantiza aislamiento: entre T₃ y las
compensaciones los asistentes pudieron ver la invitación; por eso las
compensaciones deben ser idempotentes y reintentables, y los pasos sin
compensación posible exigen aprobación previa.


In [ ]:
result = run_lab("workflow", seed=158)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


## Reflexión

1. Un equipo añadió reintentos «para robustez» a una llamada que crea pedidos y
   aparecieron pedidos duplicados: ¿qué faltó exactamente y en qué lado
   (cliente o servidor) se implementa?
2. Durante una caída parcial del proveedor de LLM, la latencia p99 de TODO tu
   sistema se multiplicó aunque la mayoría de peticiones no usaban el LLM:
   ¿qué antipatrón de Nygard describe esto y qué patrón lo corta?
3. ¿Por qué el rollback del prompt a la versión anterior puede no recuperar la
   calidad, y qué evidencia (clases 153-156) te permitiría distinguir la causa
   en horas y no en semanas?
